<!-- nav -->
[← 8.06 · TypeScript RegExp](08_06_TypeScript_RegExp.ipynb) · [Home](../../README.md) · [8.08 · C Preprocessor →](08_08_C_Preprocessor.ipynb)

# 8.07 · C Types

C writes a type where an expression needs one: in a cast, `(uint8_t)200`, in `sizeof(int)`, in a compound literal,
`(int){1}`. What it writes there is a **type name** (ISO/IEC 9899, 6.7.7, "Type names"): a list of specifiers and
qualifiers, `const char`, then an abstract declarator, `*`, that says how the type is built from them. Tutorial 8
showed that a type name is a term of a dialect of its own, `C Types` (08.1); this page builds its terms from the
smallest up, the way tutorials 3 to 7 built expressions: a keyword, then specifiers and declarators, then structures
and enumerations, each printed back as C writes it; then what validation says of them, how C reads one in a cast, and
the editions they come from.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `S` is its holder
namespace, `store.Expressions`, and `amd64` x86-64's System V ABI, which a C family's holder names (tutorial 2).
`variable` writes a name in C's terms (tutorial 4), and `keyword` a type specifier that is a keyword, such as `int`,
each a spec the builders call, as tutorial 8's `uint8` is.

In [1]:
from mbse.Expressions.C import Expressions as C, Types as CTypes
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
amd64 = store.Instances.Expressions.SystemV.AMD64
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))
keyword = lambda word: lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))
DIALECT = CTypes.Expressions.DIALECT


def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return result.evaluate() if hasattr(result, "evaluate") else result
    except ValueError as error:
        return str(error)

<a id="T08.07.1"></a>
## 08.07.1 · A type name, `int`

```text
int
```

The smallest type name is one keyword. `C Types`' kind for a type name is `typeid`, as mbse-programs and C++'s grammar
call it: its `specifiers`, a list, and its `declarator`, absent here. A specifier is not a term but a **helper** (tutorial
6), a value its term holds: `int` is a `primitivetypespecifier` whose `keyword` is `int`. The dialect's holder,
`S.C.Types.Expression`, holds the type name as its root and names its ABI, as C's holder does, since what `int` is
depends on it (02.3). Its `Text.ToText`, `CTypes.Text.ToText`, prints it as C writes it:

In [2]:
with store.transaction():
    int_ = S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([keyword("int")]))).abi([amd64]))
term = Terms.term(int_)
form = Terms.form(term)
print(CTypes.Text.ToText(int_), "|", Terms.kind_of(term)[0], "of", Terms.dialect_of(term))
print(form.kind, "| specifiers:", *form.attributes["specifiers"], "| its terms:", len(form.arguments))
print(DIALECT.name(), DIALECT.tag, "| its kinds:", ", ".join(DIALECT.kinds()))
assert CTypes.Text.ToText(int_) == "int" and [entry.abi for entry in int_.abi] == [amd64]

int | typeid of Expressions.C.Types.Term
typeid | specifiers: Part(kind='primitivetypespecifier', attributes={'keyword': 'int'}, size=0) | its terms: 0
C Types c_types | its kinds: typeid, enumerator, error, missing


Its kinds are two, `typeid` and `enumerator` (08.07.3), with `error` and `missing`, which every complete dialect has for
text it could not read (08.07.4). Everything else in a type name is a helper of `typeid`'s, twenty-two of them, each a part
of C's grammar for type names:

In [3]:
print(", ".join(CTypes.Expressions.HELPERS))

DeclSpecifier, CvQualifier, PrimitiveTypeSpecifier, NamedTypeSpecifier, ClassSpecifier, EnumSpecifier, PlaceholderTypeSpecifier, TypeofSpecifier, AtomicTypeSpecifier, BitIntSpecifier, IdDeclarator, PointerDeclarator, ArrayDeclarator, FunctionDeclarator, ParenthesizedDeclarator, ParameterDeclaration, EllipsisParameter, StandardAttributeSpecifier, Attribute, AlignasSpecifier, GnuAttributeSpecifier, DeclspecSpecifier


<a id="T08.07.2"></a>
## 08.07.2 · Specifiers, then declarators

```text
unsigned long        const char*        char[4]        int(*)[4]        int(*)(int, ...)
```

Specifiers come first, in order, each a helper: two keywords, `unsigned long`; a qualifier, `const`, and a keyword. A
**declarator** follows them, a helper too, and nests as C's declarators do, from the outside in: `const char*` is a
`pointerdeclarator`; `char[4]` an `arraydeclarator`, whose `size` is C's term, the literal `4`; `int(*)[4]`, a pointer to
an array of four `int`s, is an `arraydeclarator` whose own declarator is the pointer, which the printer parenthesizes;
and `int(*)(int, ...)`, a pointer to a function, a `functiondeclarator` whose parameters are helpers again. A form reads
a type name as it reads any term (tutorial 6): its helpers among its attributes, and the terms they hold, such as an
array's size, its arguments, each a term of the dialect that reads it:

In [4]:
pointer = lambda d: d.pointerdeclarator(lambda p: p)
four = lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(4)))
with store.transaction():
    names = [S.C.Types.Expression(lambda e, spec=spec: e.root(lambda u: u.typeid(spec)).abi([amd64])) for spec in [
        lambda i: i.specifiers([keyword("unsigned"), keyword("long")]),
        lambda i: i.specifiers([lambda s: s.cvqualifier(lambda q: q.keyword("const")), keyword("char")]).declarator(pointer),
        lambda i: i.specifiers([keyword("char")]).declarator(lambda d: d.arraydeclarator(lambda a: a.size(four))),
        lambda i: i.specifiers([keyword("int")]).declarator(lambda d: d.arraydeclarator(lambda a: a.declarator(pointer).size(four))),
        lambda i: i.specifiers([keyword("int")]).declarator(lambda d: d.functiondeclarator(lambda f: f
            .declarator(lambda p: p.parenthesizeddeclarator(lambda q: q.declarator(pointer)))
            .parameters([lambda p: p.parameterdeclaration(lambda q: q.specifiers([keyword("int")])),
                         lambda p: p.ellipsisparameter(lambda e: e)])))]]
for name in names:
    form = Terms.form(Terms.term(name))
    declarator = form.attributes.get("declarator")
    print(f"{CTypes.Text.ToText(name):17}", "specifiers:", ", ".join(s.kind for s in form.attributes["specifiers"]),
          "| declarator:", declarator.kind if declarator else "-",
          "| its terms:", ", ".join(f"{Terms.kind_of(a)[0]} of {Terms.dialect_of(a)}" for a in form.arguments) or "none")
assert [CTypes.Text.ToText(name) for name in names] == [
    "unsigned long", "const char*", "char[4]", "int(*)[4]", "int(*)(int, ...)"]

unsigned long     specifiers: primitivetypespecifier, primitivetypespecifier | declarator: - | its terms: none
const char*       specifiers: cvqualifier, primitivetypespecifier | declarator: pointerdeclarator | its terms: none
char[4]           specifiers: primitivetypespecifier | declarator: arraydeclarator | its terms: literal of Expressions.C.Expressions.Term
int(*)[4]         specifiers: primitivetypespecifier | declarator: arraydeclarator | its terms: literal of Expressions.C.Expressions.Term
int(*)(int, ...)  specifiers: primitivetypespecifier | declarator: functiondeclarator | its terms: none


<a id="T08.07.3"></a>
## 08.07.3 · Structures and enumerations

```text
struct S {}        enum Color {
                       RED = 1,
                       GREEN,
                       BLUE
                   }
```

A structure's or an enumeration's specifier may declare it, with a body: `struct S {}` is a `classspecifier` (its `key`
`struct`, or `union`) whose `body` is set. A structure's members are declarations, mbse-programs' statements, which a
type name does not hold. An enumeration's are its **enumerators**, each a term of `C Types`' own, its kind `enumerator`:
a name, and a `value`, C's term, where one is written. So `enum Color { RED = 1, GREEN, BLUE }` is a `typeid` whose
`enumspecifier` holds three `enumerator`s, the first holding the literal `1`. The printer writes the body's items a
line each, as mbse-programs does; `Terms.walk` reads the tree, parents first:

In [5]:
enumerator = lambda name, value=None: lambda t: t.enumerator(
    lambda r: r.name(lambda i: i.spelling(name)) if value is None else r.name(lambda i: i.spelling(name)).value(value))
one = lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(1)))
with store.transaction():
    structure = S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([
        lambda s: s.classspecifier(lambda c: c.key("struct").name(lambda n: n.identifier(lambda i: i.spelling("S"))).body(True))]))
    ).abi([amd64]))
    color = S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([
        lambda s: s.enumspecifier(lambda n: n.key("enum").name(lambda m: m.identifier(lambda i: i.spelling("Color"))).body(True)
            .enumerators([enumerator("RED", one), enumerator("GREEN"), enumerator("BLUE")]))]))).abi([amd64]))
print(CTypes.Text.ToText(structure))
print(CTypes.Text.ToText(color))
for t in Terms.walk(color):
    print(" ", f"{Terms.kind_of(t)[0]:10}", Terms.dialect_of(t), *([Terms.name_of(t)] if Terms.name_of(t) else []))
assert CTypes.Text.ToText(structure) == "struct S {}"
assert CTypes.Text.ToText(color) == "enum Color {\n    RED = 1,\n    GREEN,\n    BLUE\n}"

struct S {}
enum Color {
    RED = 1,
    GREEN,
    BLUE
}
  typeid     Expressions.C.Types.Term
  enumerator Expressions.C.Types.Term RED
  literal    Expressions.C.Expressions.Term
  enumerator Expressions.C.Types.Term GREEN
  enumerator Expressions.C.Types.Term BLUE


An enumerator is a term of its own, so its holder may hold one alone, and it prints as it stands in a body. A body may
hold more than enumerators: the preprocessor's directives and comments stand among them, which
[8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb) builds, so `C Types`' `Term` lists `C Preprocessor`'s directives
besides its own kinds (08.3 of tutorial 8):

In [6]:
with store.transaction():
    alone = S.C.Types.Expression(lambda e: e.root(enumerator("RED", one)).abi([amd64]))
print(CTypes.Text.ToText(alone))
print(", ".join(b.name for b in store.registered(DIALECT.union).branches))
assert CTypes.Text.ToText(alone) == "RED = 1"

RED = 1
typeid, enumerator, error, missing, includedirective, definedirective, ifdirective, ifdefdirective, elifdirective, elifdefdirective, elsedirective, otherdirective, comment


<a id="T08.07.4"></a>
## 08.07.4 · Checked, and kept as written

`DIALECT.validate` checks a type name as C's grammar has it, and reports what is wrong by path, without evaluating: an
enumerator's name is an identifier, so `1A` is refused; a body's enumerators need the body (`enum E` declares none).
The terms a type name holds are C's, validated as C validates them, so an array's size or an enumerator's value that
reads `x` needs `x` from its scope (tutorial 4), as `Symbolics.free` says. `C Types` binds no name of its own: an
enumerator's name is declared, not bound within the term:

In [7]:
with store.transaction():
    checked = {
        "1A": S.C.Types.Expression(lambda e: e.root(enumerator("1A")).abi([amd64])),
        "enum E": S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([
            lambda s: s.enumspecifier(lambda n: n.key("enum").name(lambda m: m.identifier(lambda i: i.spelling("E")))
                .enumerators([enumerator("A")]))]))).abi([amd64])),
        "int[x + 1]": S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([keyword("int")])
            .declarator(lambda d: d.arraydeclarator(lambda a: a.size(lambda t: t.binaryexpression(lambda b: b
                .left(variable("x")).operator("+").right(one))))))).abi([amd64])),
    }
for source, name in checked.items():
    print(f"{source:11}", "|", "; ".join(DIALECT.validate(name)) or "valid",
          "| needs:", ", ".join(sorted(Symbolics.free(name, DIALECT))) or "nothing")
assert DIALECT.validate(checked["1A"]) == ["'1A' is not an identifier"]
assert DIALECT.validate(checked["int[x + 1]"], {"x"}) == []

1A          | '1A' is not an identifier | needs: nothing
enum E      | an EnumSpecifier with enumerators has a body | needs: nothing
int[x + 1]  | declarator size: left: variable 'x' is not bound | needs: x


A type name read from text that does not parse is kept as written ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):
`int int` is no type name, and a reader that meets it writes `C Types`' `error`, with the text it skipped, which prints
back as written and which validation reports, here where a cast holds it:

In [8]:
with store.transaction():
    skipped = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.castexpression(lambda r: r
        .type(lambda t: t.error(lambda x: x.text("int int"))).operand(variable("x")))).abi([amd64]))
print(C.Text.ToText(skipped), "|", "; ".join(C.Expressions.DIALECT.validate(skipped, {"x"})))
assert C.Expressions.DIALECT.validate(skipped, {"x"}) == ["type: invalid syntax 'int int'"]

(int int)x | type: invalid syntax 'int int'


<a id="T08.07.5"></a>
## 08.07.5 · Read by C, in a cast

```text
(uint8_t)(x + 1)        (unsigned char)300        sizeof(int)
```

C reads `C Types`' terms where its grammar puts a type, through the dialects it reads (tutorial 8, 08.1): a cast's
`type`, a compound literal's, a generic association's; and `typeid` stands among C's own kinds, where an expression or a
type may, in `sizeof`. So C validates, infers and simplifies a cast with its type in it: with `x` 255, `(uint8_t)(x + 1)` folds its operand to
`256` and keeps the cast, since C writes no literal of a `uint8_t` (where the value is used, the ABI's `uint8_t` wraps it
to 0); `(unsigned char)300` stays as written for the same reason, and C infers its domain from the type it names, the
ABI's `unsigned char`. What reads a type is the cast that holds it: C has no rule for `sizeof`, nor for a type name
alone, and each stays in the residual, named by the dialect whose kind it is:

In [9]:
uint8 = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(
    lambda n: n.name(lambda m: m.identifier(lambda d: d.spelling("uint8_t"))))]))
with store.transaction():
    wrapped = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.castexpression(lambda r: r.type(uint8)
        .operand(lambda o: o.binaryexpression(lambda b: b.left(variable("x")).operator("+").right(one))))).abi([amd64]))
    narrowed = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.castexpression(lambda r: r
        .type(lambda t: t.typeid(lambda i: i.specifiers([keyword("unsigned"), keyword("char")])))
        .operand(lambda o: o.literal(lambda l: l.value(lambda v: v.as_scalar(300)))))).abi([amd64]))
    sized = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.sizeofexpression(
        lambda r: r.operand(lambda t: t.typeid(lambda i: i.specifiers([keyword("int")]))))).abi([amd64]))
def residual(expression, scope):
    """What simplifying `expression` in `scope` leaves, as C writes it: a literal, or the term with what it decides
    folded."""
    with store.transaction():
        left = S.C.Expressions.Expression(lambda e: e.root(C.Simplifiers.OfAny(expression, scope).residual).abi([amd64]))
    return C.Text.ToText(left)


for expression in (wrapped, narrowed, sized):
    type_ = Terms.form(Terms.term(expression)).arguments[0]  # each one's first subterm, its type
    print(f"{C.Text.ToText(expression):18}", f"{Terms.kind_of(Terms.term(expression))[0]:16}",
          "its type:", Terms.kind_of(type_)[0], "of", C.Expressions.DIALECT.owner(type_).name(),
          "| x = 255:", residual(expression, {"x": 255}))
print("C infers", C.Text.ToText(narrowed), "as", C.Expressions.DIALECT.infer(narrowed).name())
print(refused(C.Simplifiers.OfAny, int_))
assert [C.Text.ToText(x) for x in (wrapped, narrowed, sized)] == ["(uint8_t)(x + 1)", "(unsigned char)300", "sizeof(int)"]
assert [residual(x, {"x": 255}) for x in (wrapped, narrowed, sized)] == ["(uint8_t)256", "(unsigned char)300", "sizeof(int)"]

(uint8_t)(x + 1)   castexpression   its type: typeid of C Types | x = 255: 0
(unsigned char)300 castexpression   its type: typeid of C Types | x = 255: 44
sizeof(int)        sizeofexpression its type: typeid of C Types | x = 255: C sizeofexpression is not evaluated
C infers (unsigned char)300 as unsigned char
C Types typeid is not evaluated


`int` is the ABI's (02.3), so in a holder that names no ABI neither the literal `300`, an `int`, nor a cast to `int`
has a domain, and inference refuses the first it meets, the cast's operand:

In [10]:
with store.transaction():
    unnamed = S.C.Expressions.Expression(lambda e: e.root(lambda u: u.castexpression(lambda r: r
        .type(lambda t: t.typeid(lambda i: i.specifiers([keyword("int")])))
        .operand(lambda o: o.literal(lambda l: l.value(lambda v: v.as_scalar(300)))))))
print(refused(C.Expressions.DIALECT.infer, unnamed))

cannot infer the domain of an invalid expression: operand: 'int' is not a type C defines, and its expression names no ABI


<a id="T08.07.6"></a>
## 08.07.6 · Its editions

`C Types` covers C's three editions, as C does (02.3), each by its designation and its common name, and records the
kinds each has ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
both of its kinds are in all three, so no kind of it is newer than C11:

In [11]:
types = store.Instances.Expressions.C.Types  # the dialect, by its global name
for designation, edition in Standards.standards(types).items():
    print(designation, "|", ", ".join(Standards.tags(edition)[1:]), "|", ", ".join(kind.name for kind in Standards.kinds(edition)))
assert Standards.find(types, "C23") is Standards.standards(types)["ISO/IEC 9899:2024"]

ISO/IEC 9899:2011 | C11 | Expressions.C.Types.TypeId, Expressions.C.Types.Enumerator
ISO/IEC 9899:2018 | C17 | Expressions.C.Types.TypeId, Expressions.C.Types.Enumerator
ISO/IEC 9899:2024 | C23 | Expressions.C.Types.TypeId, Expressions.C.Types.Enumerator


## What you learned

- **A type name is a `C Types` term**, `typeid`: its specifiers and its declarator are helpers, nested as C's
  declarators nest, and the terms they hold, an array's size, an enumerator's value, are C's.
- **An enumeration's enumerators are terms of their own**, `enumerator`, among which the preprocessor's directives may
  stand ([8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb)).
- **Its holder names an ABI**, as C's does, since what `int` is depends on it.
- **It is checked and kept as written**: validation reports what C's grammar refuses, and text that does not parse is
  kept as `error`, printed back as written.
- **C reads it where a type stands**: a cast converts by it and infers its domain from it, while a type name itself is never folded.

C++'s type-ids are a dialect of their own, [8.09 · Cpp Types](08_09_Cpp_Types.ipynb); next comes the preprocessor that C
and C++ share.

<!-- nav -->
[← 8.06 · TypeScript RegExp](08_06_TypeScript_RegExp.ipynb) · [Home](../../README.md) · [8.08 · C Preprocessor →](08_08_C_Preprocessor.ipynb)